# PDF 파서 파이프라인
특허/논문 PDF → Markdown 변환 + Gemini Vision API 수식 보완

In [ ]:
# 셀 1 — 환경 세팅 (세션마다 실행)
import os
!apt-get install -y openjdk-17-jdk-headless -qq
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!pip install "opendataloader-pdf[hybrid]" pymupdf google-generativeai -q
!java -version
print("✓ 완료")

In [ ]:
# 셀 2 — output 초기화 + PDF 업로드 (공통, 항상 실행)
import os, shutil
from google.colab import files

# 이전 결과 초기화
if os.path.exists("./output"):
    shutil.rmtree("./output")
os.makedirs("./output")
print("output 폴더 초기화 완료")

uploaded = files.upload()  # Ctrl+클릭으로 다중 선택 가능

# 파일명 공백/괄호 제거 (CLI 파싱 오류 방지)
pdf_files = []
for fname in uploaded.keys():
    clean = fname.replace(" ", "_").replace("(", "").replace(")", "")
    if fname != clean:
        os.rename(fname, clean)
    pdf_files.append(clean)

print(f"업로드된 파일 {len(pdf_files)}개: {pdf_files}")
print("→ 아래 셀 2-a 또는 2-b 중 하나만 실행하세요")

### 파서 선택: 셀 2-a / 2-b / 2-c 중 하나만 실행
| | 셀 2-a (opendataloader) | 셀 2-b (pymupdf) | 셀 2-c (LlamaParse) |
|---|---|---|---|
| 속도 | 느림 (파일당 3~5분) | 빠름 (파일당 수초) | 빠름 (병렬 처리) |
| 품질 | 좋음 | 단순 (페이지 단위) | 좋음 |
| JSON 출력 | O (셀 4 수식 보완 가능) | X | X |
| API 키 필요 | X | X | O (LlamaIndex) |
| 추천 상황 | 수식 보완까지 할 때 | 빠르게 텍스트만 | 품질+속도 균형 |

In [ ]:
# 셀 2-a — PDF 파싱 (opendataloader, 느리지만 정확)
# 수식 보완(셀 3~5)이 필요한 경우 이 셀을 실행
import opendataloader_pdf

opendataloader_pdf.convert(
    input_path=pdf_files,
    output_dir="./output",
    format="markdown,json"
)
print("파싱 완료")

In [ ]:
# 셀 2-b — PDF 파싱 (pymupdf, 빠르고 단순)
# 텍스트만 빠르게 뽑을 때 이 셀을 실행. 수식 보완 불가 (셀 3~5 건너뜀)
import fitz, os

for pdf in pdf_files:
    doc = fitz.open(pdf)
    pages = []
    for i, page in enumerate(doc, 1):
        text = page.get_text().strip()
        if text:
            pages.append(f"<!-- page {i} -->\n\n{text}")
    doc.close()

    out = f"./output/{os.path.splitext(pdf)[0]}.md"
    with open(out, "w") as f:
        f.write("\n\n---\n\n".join(pages))
    print(f"완료: {out}")

print("파싱 완료 → 셀 2-c 실행해서 다운로드")

In [ ]:
# 셀 2-c — PDF 파싱 (LlamaParse, 병렬 처리)
# 텍스트 품질이 pymupdf보다 좋고, 여러 파일을 동시에 처리함. 수식 보완 불가 (셀 3~5 건너뜀)
# API 키 등록: 왼쪽 사이드바 🔑 아이콘 → 이름: LLAMA_API_KEY, 값: 키 입력
!pip install llama-cloud nest_asyncio -q

import nest_asyncio
nest_asyncio.apply()  # Colab 비동기 이벤트 루프 충돌 방지

from llama_cloud_services import LlamaParse
from google.colab import userdata
import os

LLAMA_API_KEY = userdata.get("LLAMA_API_KEY")

# 품질 모드 선택 (하나만 주석 해제)
# 웹 UI "fast"          → 가장 빠름, txt 출력
PARSE_MODE = "parse_page_without_llm"; RESULT_TYPE = "text";     EXT = ".txt"
# 웹 UI "cost effective" → 텍스트 위주, 경제적, md 출력
# PARSE_MODE = "parse_page_with_llm";    RESULT_TYPE = "markdown"; EXT = ".md"
# 웹 UI "agentic"        → 다이어그램/이미지 포함 문서
# PARSE_MODE = "parse_page_with_agent";  RESULT_TYPE = "markdown"; EXT = ".md"; MODEL = "openai-gpt-4-1-mini"
# 웹 UI "agentic plus"   → 복잡한 레이아웃, 표, 수식 (크레딧 많이 소모)
# PARSE_MODE = "parse_page_with_layout_agent"; RESULT_TYPE = "markdown"; EXT = ".md"; MODEL = "anthropic-sonnet-4.0"

MODEL = None  # agentic / agentic plus 사용 시 위에서 MODEL 직접 지정

parser = LlamaParse(
    api_key=LLAMA_API_KEY,
    result_type=RESULT_TYPE,
    parse_mode=PARSE_MODE,
    **({"model": MODEL} if MODEL else {}),
    num_workers=4,
    verbose=True,
)
print(f"모드: {PARSE_MODE} / 출력: {RESULT_TYPE}")

failed = []
for pdf in pdf_files:
    try:
        docs = parser.load_data(pdf)
        if not docs:
            raise ValueError("빈 결과 반환")
        out = f"./output/{os.path.splitext(pdf)[0]}{EXT}"
        with open(out, "w") as f:
            f.write("\n\n---\n\n".join(doc.text for doc in docs))
        print(f"완료: {out}")
    except Exception as e:
        print(f"[실패] {pdf}: {e}")
        failed.append(pdf)

if failed:
    print(f"\n실패한 파일 {len(failed)}개: {failed}")
else:
    print("\n모든 파일 파싱 완료 → 셀 2-d 실행해서 다운로드")

In [ ]:
# 셀 2-d — [빠른 다운로드] 수식 보완 없이 zip으로 다운로드
# 2-a / 2-b / 2-c 실행 후, 수식 보완이 필요 없으면 이 셀 실행 후 종료. 셀 3~5는 건너뜀.
import os, zipfile
from google.colab import files

zip_path = "./output/result.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for pdf in pdf_files:
        base = os.path.splitext(pdf)[0]
        for ext in (".md", ".txt"):
            path = f"./output/{base}{ext}"
            if os.path.exists(path):
                zf.write(path, os.path.basename(path))
                print(f"추가: {path}")
                break
        else:
            print(f"[오류] 파일 없음: {base}.md / .txt")

files.download(zip_path)
print("다운로드 완료")

In [ ]:
# 셀 3 — JSON 구조 확인 (셀 4 실행 전 반드시 확인)
import json, os

# pdf_files[0] 기준 자동 경로 설정
base_name = os.path.splitext(pdf_files[0])[0]
JSON_PATH = f"./output/{base_name}.json"

with open(JSON_PATH) as f:
    data = json.load(f)

# 최상위 키 확인
print("=== 최상위 키 ===")
print(list(data.keys()))

# elements 키 존재 여부 및 첫 5개 확인
elements = data.get("elements", data.get("blocks", data.get("content", [])))
print(f"\n=== elements 후보 키로 찾은 항목 수: {len(elements)} ===")

if elements:
    print("\n=== 첫 번째 element 구조 ===")
    print(json.dumps(elements[0], indent=2, ensure_ascii=False))

    # 실제 사용된 키 집합 확인
    all_keys = set()
    all_types = set()
    for el in elements:
        all_keys.update(el.keys())
        if "type" in el:
            all_types.add(el["type"])
    print(f"\n=== element에 쓰인 모든 키: {sorted(all_keys)} ===")
    print(f"=== type 값 종류: {sorted(all_types)} ===")
else:
    print("\n[주의] elements 키를 찾지 못했습니다. 전체 구조를 확인하세요:")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

In [ ]:
# 셀 4 — 수식 보완 (Gemini Vision API)
# 셀 3 실행 후 ELEMENTS_KEY / TYPE_KEY / PAGE_KEY / BBOX_KEY 를 실제 키로 수정할 것
# API 키 등록: 왼쪽 사이드바 🔑 아이콘 → 이름: GEMINI_API_KEY, 값: 키 입력
import json, fitz, base64, re, os
import google.generativeai as genai
from google.colab import userdata

PDF_PATH  = pdf_files[0]
JSON_PATH = f"./output/{os.path.splitext(pdf_files[0])[0]}.json"
MD_PATH   = f"./output/{os.path.splitext(pdf_files[0])[0]}.md"

# 모델 선택 (하나만 주석 해제)
MODEL = "gemini-2.5-flash"   # 추천: 속도·정확도 균형
# MODEL = "gemini-2.5-pro"   # 복잡한 수식이 많을 때
# MODEL = "gemini-2.0-flash" # 빠르고 저렴하게

# ↓ 셀 3 결과에 맞게 수정
ELEMENTS_KEY = "elements"  # 최상위에서 element 목록을 담는 키
TYPE_KEY     = "type"      # element 내 타입 키
PARA_TYPE    = "paragraph" # 단락을 나타내는 type 값
TEXT_KEY     = "text"      # 텍스트 내용 키
PAGE_KEY     = "page"      # 페이지 번호 키
BBOX_KEY     = "bbox"      # 바운딩박스 키 ([x0,y0,x1,y1])

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
model = genai.GenerativeModel(MODEL)
print(f"모델: {MODEL}")

with open(JSON_PATH) as f:
    data = json.load(f)

doc = fitz.open(PDF_PATH)

def crop_to_bytes(page_num, bbox):
    page = doc[page_num - 1]
    rect = fitz.Rect(bbox[0], bbox[1], bbox[2], bbox[3])
    clip = page.get_pixmap(matrix=fitz.Matrix(2, 2), clip=rect)
    return clip.tobytes("png")

def to_latex(img_bytes):
    resp = model.generate_content([
        {"mime_type": "image/png", "data": base64.b64encode(img_bytes).decode()},
        "이 이미지의 수식을 LaTeX로 변환해줘. $$ $$ 블록으로 감싸서 수식만 출력해. 설명 없이."
    ])
    return resp.text.strip()

elements  = data.get(ELEMENTS_KEY, [])
latex_map = {}

for el in elements:
    if el.get(TYPE_KEY, "") != PARA_TYPE:
        continue
    text = el.get(TEXT_KEY, "")
    if len(text) < 100 and re.search(r'[¼½¾∈ℜ∑∂⁎αβγηΦ]|argm|ð\d\)', text):
        page = el.get(PAGE_KEY, 1)
        bbox = el.get(BBOX_KEY, [])
        if not bbox:
            continue
        img_bytes = crop_to_bytes(page, bbox)
        latex     = to_latex(img_bytes)
        latex_map[(page, tuple(bbox))] = (text, latex)
        print(f"p.{page}: {text[:50]} → {latex[:60]}")

doc.close()
print(f"총 {len(latex_map)}개 수식 변환 완료")

In [ ]:
# 셀 5 — MD 교체 & 다운로드
from google.colab import files

with open(MD_PATH) as f:
    md = f.read()

for (page, bbox), (original, latex) in latex_map.items():
    if original in md:
        md = md.replace(original, latex)

out_path = MD_PATH.replace(".md", "_fixed.md")
with open(out_path, "w") as f:
    f.write(md)

print(f"저장 완료: {out_path}")
files.download(out_path)